In [1]:
%reset -f

In [2]:
typ = "example"  # Type of simulation, used for naming files

In [3]:

import xml.etree.ElementTree as ET
import mujoco
import csv
import open3d as o3d
from sim_fxn_lib import *
import numpy as np
import os
import sys
import pickle
import scipy


In [4]:
v = {}

In [5]:
# load in the mujoco model
xml_path = 'RHex1-water-rigid.xml'
tMax = 3
dt = 0.001
model, data, renderer, t, dt, frames, framerate, sand_h_id, stl_path = initialize_simulation(xml_path=xml_path, stl='sandflipper.stl', tMax=tMax, dt=dt)
numSteps = len(t)
tMax = t[-1]
renderer.update_scene(data, camera="diag") # can replace with other cameras
scene_option = mujoco.MjvOption() # these two scene lines make the contact forces visible in the sim
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True


In [6]:
# this grabs the mujoco IDs for the position controller actuators

pos_actuator_ids = {}
actuator_names = [
    "front right_p", "front left_p",
    "middle right_p", "middle left_p",
    "back right_p", "back left_p"
]

# Loop through each name and get the corresponding ID
for name in actuator_names:
    pos_actuator_ids[name] = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)

fr_pos_id = pos_actuator_ids["front right_p"]
mr_pos_id = pos_actuator_ids["middle right_p"]
br_pos_id = pos_actuator_ids["back right_p"]
fl_pos_id = pos_actuator_ids["front left_p"]
ml_pos_id = pos_actuator_ids["middle left_p"]
bl_pos_id = pos_actuator_ids["back left_p"]
# distal_fr_vert_pos_id = pos_actuator_ids["distal segment front right_vert"]

print("Actuator IDs:")
for name, actuator_id in pos_actuator_ids.items():
    print(f"{name}: {actuator_id}")

Actuator IDs:
front right_p: 0
front left_p: 3
middle right_p: 1
middle left_p: 4
back right_p: 2
back left_p: 5


In [7]:
# --------ACTUAL CODE -------------
# to measure displacement
plate_id = mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_BODY,"plate")
start_pos = data.xpos[plate_id].copy()

tripod_left = ["mid right" , "front left" , "back left"]
 
joint_id_mr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[0])

joint_id_fl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[1])
    
joint_id_bl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[2])
    
commanded_left_angle = 0.0
    
# -------- RIGHT TRIPOD GAIT ---------
tripod_right = ["mid left" , "front right" , "back right"]

joint_id_ml = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[0])
    
joint_id_fr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[1])

joint_id_br = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[2])

commanded_right_angle = -160*(np.pi/180)

In [8]:
body, vertices, faces, mesh = load_and_process_mesh(stl_path, scale_factor=1000)

In [9]:
entities = get_named_bodies_from_xml(xml_path)

In [10]:
for entity in entities:
    v[f'fm_{entity}'] = []
    v[f'mm_{entity}'] = []

motion_data = {
    "time": [],
    "x": [],
    "y": [],
    "z": [],
    "roll": [],
    "pitch": [],
    "yaw": []
}

In [11]:
camera_list = ["diag"]
frames = {cam: [] for cam in camera_list}
combined_framesFR = []
frames_diag = []

In [12]:
i = 0

In [13]:
import mujoco
import mujoco.viewer
import time
import numpy as np
import scipy.spatial.transform
applied_force = True
mass = {}
save_every = 8
for body_name in entities:
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)

    mass[body_name] = model.body_mass[body_id]

import os, cv2                      # ➊ use OS & OpenCV -Sophia
os.makedirs("frames", exist_ok=True) # ➋ ensure output folder exists -Sophia

# Reset simulation
model.opt.timestep = dt  # match external dt
once_submerged = False  # Flag to check if any body has submerged
# with mujoco.viewer.launch_passive(model, data) as viewer:
#print(len(t))

plate_pos = []


objc[83566]: Class GLFWHelper is implemented in both /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/open3d/cpu/pybind.cpython-311-darwin.so (0x143fe7a28) and /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/glfw/libglfw.3.dylib (0x1689adbb8). One of the two will be used. Which one is undefined.
objc[83566]: Class GLFWApplicationDelegate is implemented in both /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/open3d/cpu/pybind.cpython-311-darwin.so (0x143fe7a78) and /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/glfw/libglfw.3.dylib (0x1689adc08). One of the two will be used. Which one is undefined.
objc[83566]: Class GLFWWindowDelegate is implemented in both /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/open3d/cpu/pybind.cpython-311-darwin.so (0x143fe7aa0) and /Users/haleybrewster/anaconda3/lib/python3.11/site-packages/glfw/libglfw.3.dylib (0x1689adc30). One of the two will be used. Which one is undefined.
objc[83566]: Class 

In [14]:
#MuJoCo time steps is every 0.01
stance_step = 0.16*(np.pi/180)
swing_step = 0.56*(np.pi/180)
left_leg_state = "swing"
        
start_stance_list = []
end_stance_list = []
    
stance_reps = 20
for cycle in range(stance_reps):
    start_stance = 160*(np.pi/180) - 2*np.pi*cycle
    end_stance = 240*(np.pi/180) -2*np.pi*cycle
    
    start_stance_list.append(start_stance)  
    end_stance_list.append(end_stance)
for i in range(len(t)):

    left_leg_state = "swing"
    right_leg_state = "swing"
    
    for j in range(len(start_stance_list)):
        start_stance = start_stance_list[j]
        end_stance = end_stance_list[j]
    
        if commanded_left_angle >= start_stance and commanded_left_angle < end_stance:
            left_leg_state = "stance"

        if commanded_right_angle >= start_stance and commanded_right_angle < end_stance:
            right_leg_state = "stance"
    
    if left_leg_state == "stance":
        commanded_left_angle -= stance_step
    else:
        commanded_left_angle -= swing_step
        
    if right_leg_state == "stance":
        commanded_right_angle -= stance_step
    else:
        commanded_right_angle -= swing_step

        
    data.ctrl[pos_actuator_ids["front left_p"]] = commanded_left_angle
    data.ctrl[pos_actuator_ids["middle right_p"]] = commanded_left_angle
    data.ctrl[pos_actuator_ids["back left_p"]] = commanded_left_angle
    
    data.ctrl[pos_actuator_ids["front right_p"]] = commanded_right_angle
    data.ctrl[pos_actuator_ids["middle left_p"]] = commanded_right_angle
    data.ctrl[pos_actuator_ids["back right_p"]] = commanded_right_angle
        
    mujoco.mj_step(model, data)

    data.qfrc_applied[:] = 0
                
    robot_pos = data.xpos[plate_id].copy() 
    dist_x = robot_pos[0] - start_pos[0]

    if i % save_every == 0:
        renderer.update_scene(data, camera="diag")
        frame = renderer.render()
        cv2.imwrite(f"frames/{typ}_rigid_frame_{i:04d}.png", frame)
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  

In [15]:
with open(f"W{typ}_plate_position.csv", "w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["X [m]", "Y [m]", "Z [m]"])  # header
    for i in range(len(plate_pos)):
        row = [f"{coord:.4f}" for coord in plate_pos[i]]
        writer.writerow(row)

In [16]:
def create_video_from_frames(typ, frame_folder, frame_prefix, save_every, output_name=None):
    fps = 1000 / save_every  # Match the simulation timestep
    if output_name is None:
        output_video = f"{typ}_rigid.mp4"
    else:
        output_video = output_name
    frames = sorted([f for f in os.listdir(frame_folder) 
                     if f.startswith(frame_prefix) and f.endswith(".png")])
    first_frame = cv2.imread(os.path.join(frame_folder, frames[0]))
    height, width, _ = first_frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
    # Write all frames to video
    for f in frames:
        img = cv2.imread(os.path.join(frame_folder, f))
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        out.write(img)
    out.release()
    print(f"Video saved to {output_video}")
    return output_video

In [17]:
create_video_from_frames(
    typ=typ,
    frame_folder=r"frames",
    frame_prefix=f"{typ}_rigid_frame_",
    save_every=save_every,
)

Video saved to example_rigid.mp4


'example_rigid.mp4'